# Program 8: A short paragraph from a seed word

**Aim.** Write a program that takes a seed word, generates similar words and constructs a short paragraph using them.

**Theory.** The program pipeline: seed -> similar words (embeddings) -> paragraph (LLM) -> check that every word appears.

In [1]:
import sys
sys.path.insert(0, '.')
from common import *


In [2]:
import re
import gensim.downloader as api
wv = api.load('glove-wiki-gigaword-100')
llm, label = get_llm(0.7)
print('model:', label)

def story(seed, n=4):
    words = [w for w, _ in wv.most_similar(seed, topn=15) if w.isalpha()][:n]
    text = ask(llm, f'Write a 4-sentence paragraph about {seed}. Use every one of these words exactly as written: ' + ', '.join(words) + '. Output the paragraph only.')
    missing = [w for w in words if not re.search(rf'\b{w}\b', text, re.I)]
    return words, text, missing

for seed in ['medicine', 'recovery']:
    words, text, missing = story(seed)
    print(f'SEED {seed}: {words}\n{text}\nwords missing from the paragraph: {missing or "none"}\n')

model: Groq openai/gpt-oss-20b (no Cohere key available)


SEED medicine: ['medical', 'medicines', 'health', 'veterinary']
The medical field constantly evolves to meet the demands of patients worldwide. Innovative medicines are developed through rigorous research and clinical trials. Maintaining health requires a balanced diet, regular exercise, and preventive care. Veterinary medicine ensures the well-being of animals, which indirectly supports human society.
words missing from the paragraph: none


SEED recovery: ['economy', 'growth', 'economic', 'slowing']
Recovery can feel like a fragile bridge over a turbulent river, where each step forward requires patience and resilience. The community's collective efforts have begun to lift the economy out of its recent downturn. This momentum is driven by renewed investment and a steady pace of growth, reflecting an economic shift toward stability. Although the process is not without challenges, the current slowing of external pressures offers a chance to rebuild stronger foundations.
words missing from the paragraph: none


## Observations

The check step matters: an LLM may skip a word, so the program reports it instead of assuming. For safety-critical text MedSaarthi uses the same idea, checking every number in the output against the input.